In [1]:
import torch

In [2]:
# t_c - data in celcius, t_u - data in some unknown units

In [3]:
t_c = [0.5,  14.0, 15.0, 28.0, 11.0,  8.0,  3.0, -4.0,  6.0, 13.0, 21.0]
t_u = [35.7, 55.9, 58.2, 81.9, 56.3, 48.9, 33.9, 21.8, 48.4, 60.4, 68.4]

In [4]:
t_c_t = torch.tensor(t_c)
t_u_t = torch.tensor(t_u)

In [5]:
t_c_t.shape, t_u_t.shape, t_c_t.ndim, t_u_t.ndim, t_c_t.stride(), t_u_t.stride()

(torch.Size([11]), torch.Size([11]), 1, 1, (1,), (1,))

In [6]:
# assume the simple model here is t_c = w * t_u + b
# the weight tells us how much a given input influences the output. The bias is what the output would be if all inputs were zero

In [7]:
# we have a model with some unknown parameters, and we need to estimate those parameters so that
# the error between predicted outputs and measured values is as low as possible. We notice that we still need to exactly 
# define a measure of the error. Such a measure, which we refer to as the loss function (or cost function), should be high if the error is high and 
# should ideally be as low as possible for a perfect match. Our optimization process should therefore aim at finding w and b so that 
# the loss function is at a minimum

In [8]:
# t_p - predicted temperatures output by our model
# We need to make sure the loss function makes the loss positive, both when t_p is greater than and 
# when it is less than the true t_c because the goal is for t_p to match t_c.
# We have a few choices, the most straightforward being |t_p - t_c| and (t_p - t_c)^2

In [9]:
# Both of the example loss functions have a clear minimum at zero and grow monotonically 
# as the predicted value moves further from the true value in either direction.
# Because the steepness of the growth also monotonically increases away from the minimum, both are convex (ДУГООБРАЗНЫЕ). 
# Since our model is linear, the loss is a function of w, and b is also convex. 
# When our loss function is convex (shaped like a bowl), we could use specialized algorithms to efficiently find the minimum. 
# However, in this chapter, we’ll focus on two common, simple loss functions that work well across a variety of problems.
# While these fundamental approaches provide an excellent foundation, we’ll explore more sophisticated and specialized loss functions
# later when we work with deep neural networks that need to capture more complex relationships in the data

In [11]:
# t_u, w, and b to be the input tensor, weight parameter, and bias parameter, respectively. 
# In our model, the parameters will be PyTorch scalars (aka zero-dimensional tensors), 
# and the product operation will use broadcasting to yield the returned tensors.
def model(t_u, w, b):
    return w * t_u + b

In [12]:
# mean square loss
# result is scalar
def loss_fn(t_p, t_c):
    square_diffs = (t_p - t_c) ** 2
    return square_diffs.mean()

In [15]:
# w and b are scalars (zero-dimensional tensors)
w = torch.ones(())
b = torch.zeros(())
w, b

(tensor(1.), tensor(0.))

In [18]:
t_p_t = model(t_u_t, w, b)
t_p

tensor([35.7000, 55.9000, 58.2000, 81.9000, 56.3000, 48.9000, 33.9000, 21.8000,
        48.4000, 60.4000, 68.4000])

In [19]:
loss = loss_fn(t_p_t, t_c_t)
loss

tensor(1763.8846)

## Broadcasting

In [ ]:
# - For each index dimension, counted from the back, if one of the operands is size 1 in that dimension, 
# PyTorch will use the single entry along this dimension with each of the entries in the other tensor along this dimension.
# - If both sizes are greater than 1, they must be the same, and element-wise matching is used.
# - If one of the tensors has more index dimensions than the other, the entirety of the other tensor will be used for each entry along these dimensions.

In [20]:
x = torch.ones(())
y = torch.ones(3,1)
z = torch.ones(1,3)
a = torch.ones(2,1,1)
x,y,z,a

(tensor(1.),
 tensor([[1.],
         [1.],
         [1.]]),
 tensor([[1., 1., 1.]]),
 tensor([[[1.]],
 
         [[1.]]]))

In [21]:
print(f"shapes: x: {x.shape}, y: {y.shape}")
print(f"        z: {z.shape}, a: {a.shape}")

shapes: x: torch.Size([]), y: torch.Size([3, 1])
        z: torch.Size([1, 3]), a: torch.Size([2, 1, 1])


In [22]:
print("x*y: ", (x*y).shape)
print("y*z: ", (y*z).shape)
print("y*z*a: ", (y*z*a).shape)

x*y:  torch.Size([3, 1])
y*z:  torch.Size([3, 3])
y*z*a:  torch.Size([2, 3, 3])


## Loss function optimization. Gradient descent

In [23]:
# central difference approximation

In [24]:
delta = 0.1
loss_rate_of_change_w = (loss_fn(model(t_u_t, w + delta, b), t_c_t) - loss_fn(model(t_u_t, w - delta, b), t_c_t)) / 2.0 * delta
loss_rate_of_change_w

tensor(45.1730)

In [25]:
learning_rate = 1e-2
learning_rate

0.01

In [26]:
w = w - learning_rate*loss_rate_of_change_w
w

tensor(0.5483)

In [27]:
loss_rate_of_change_b = (loss_fn(model(t_u_t, w, b+delta), t_p_t) - loss_fn(model(t_u_t, w, b-delta), t_p_t)) / 2.0*delta
loss_rate_of_change_b

tensor(-0.4680)

In [28]:
b = b - learning_rate*loss_rate_of_change_b
b

tensor(0.0047)

In [ ]:
# direvative of loss - how the loss changes when a prediction changes

In [29]:
# This function returns a tensor where each element represents how much the loss would change if we slightly adjusted the corresponding prediction. 
# The factor of 2 comes from the power rule for differentiation, 
# while the division by t_p.size(0) accounts for the averaging operation in the original loss function.
def dloss_fn(t_p, t_c):
    dsq_diffs = 2*(t_p - t_c) / t_p.size(0) # The division using t_p.size(0) is the number of elements for t_p.
    return dsq_diffs

In [30]:
dloss_fn(t_p_t, t_c_t)

tensor([6.4000, 7.6182, 7.8545, 9.8000, 8.2364, 7.4364, 5.6182, 4.6909, 7.7091,
        8.6182, 8.6182])

### Applying the derivatives to the model

In [31]:
# derivatives of the model:
# with respect to w
# and with respect to b

In [32]:
def dmodel_dw(t_u, w, b):
    return t_u

In [33]:
def dmodel_db(t_u, w, b):
    return 1

### Gradient function

In [34]:
# the function returning the gradient of the loss with respect to w and b is

In [35]:
def grad_fn(t_u, t_c, t_p, w, b):
    dloss_dtp = dloss_fn(t_p, t_c)
    dloss_dw = dloss_dtp * dmodel_dw(t_u, w, b)
    dloss_db = dloss_dtp * dmodel_db(t_u, w, b)
    return torch.stack([dloss_dw.sum(), dloss_db.sum()])

### Iterating to fit the model

In [36]:
# epoc vs training iteration (step)
# In deep learning, an epoch refers to a complete pass through the entire training dataset, 
# where all samples have been used once to update the model parameters. 
# An epoch is different from a single training iteration (or step), 
# which typically involves processing one batch of data through forward and backward passes. 
# With our current implementation, we’re using the entire dataset in each iteration, so one iteration equals one epoch. 
# In larger datasets, we’ll typically split data into batches, requiring multiple iterations to complete a single epoch

In [45]:
def training_loop(t_u, t_c, params, learning_rate, nepoch):
    for epoch in range(1, nepoch+1):
        w,b = params
        t_p = model(t_u, w, b) # forward pass
        loss = loss_fn(t_p, t_c)
        grad = grad_fn(t_u, t_c, t_p, w, b) # backward pass
        print('Epoch : %d, Loss: %f, params: %s, grad: %s' % (epoch, float(loss), params, grad)) 
        params = params - learning_rate * grad   

In [46]:
# we see overfitting
training_loop(
    t_u=t_u_t,
    t_c=t_c_t,
    params=torch.tensor([1.0, 0.0]),
    learning_rate=1e-2,
    nepoch=100
)

Epoch : 1, Loss: 1763.884644, params: tensor([1., 0.]), grad: tensor([4517.2964,   82.6000])
Epoch : 2, Loss: 5802484.500000, params: tensor([-44.1730,  -0.8260]), grad: tensor([-261257.4062,   -4598.9707])
Epoch : 3, Loss: 19408033792.000000, params: tensor([2568.4011,   45.1637]), grad: tensor([15109614.0000,   266155.6875])
Epoch : 4, Loss: 64915909902336.000000, params: tensor([-148527.7344,   -2616.3931]), grad: tensor([-8.7385e+08, -1.5393e+07])
Epoch : 5, Loss: 217130559820791808.000000, params: tensor([8589999.0000,  151310.8906]), grad: tensor([5.0539e+10, 8.9023e+08])
Epoch : 6, Loss: 726257512784183951360.000000, params: tensor([-4.9680e+08, -8.7510e+06]), grad: tensor([-2.9229e+12, -5.1486e+10])
Epoch : 7, Loss: 2429183416467662896627712.000000, params: tensor([2.8732e+10, 5.0610e+08]), grad: tensor([1.6904e+14, 2.9776e+12])
Epoch : 8, Loss: 8125122549611731432050262016.000000, params: tensor([-1.6617e+12, -2.9270e+10]), grad: tensor([-9.7764e+15, -1.7221e+14])
Epoch : 9, L

### hyperparameter tuning

In [47]:
# lets decrease a learning rate from 1e-2 to 1e-4

In [48]:
# now there’s another problem: the updates to parameters are very small, so the loss decreases very slowly and eventually stalls. 
# We could obviate this problem by making learning_rate adaptive—that is, change according to the magnitude of updates. 
# There are optimization schemes that do that
training_loop(
    t_u=t_u_t,
    t_c=t_c_t,
    params=torch.tensor([1.0, 0.0]),
    learning_rate=1e-4,
    nepoch=100
)

Epoch : 1, Loss: 1763.884644, params: tensor([1., 0.]), grad: tensor([4517.2964,   82.6000])
Epoch : 2, Loss: 323.090546, params: tensor([ 0.5483, -0.0083]), grad: tensor([1859.5492,   35.7843])
Epoch : 3, Loss: 78.929634, params: tensor([ 0.3623, -0.0118]), grad: tensor([765.4666,  16.5122])
Epoch : 4, Loss: 37.552845, params: tensor([ 0.2858, -0.0135]), grad: tensor([315.0790,   8.5787])
Epoch : 5, Loss: 30.540283, params: tensor([ 0.2543, -0.0143]), grad: tensor([129.6733,   5.3127])
Epoch : 6, Loss: 29.351158, params: tensor([ 0.2413, -0.0149]), grad: tensor([53.3495,  3.9682])
Epoch : 7, Loss: 29.148882, params: tensor([ 0.2360, -0.0153]), grad: tensor([21.9303,  3.4148])
Epoch : 8, Loss: 29.113848, params: tensor([ 0.2338, -0.0156]), grad: tensor([8.9964, 3.1869])
Epoch : 9, Loss: 29.107145, params: tensor([ 0.2329, -0.0159]), grad: tensor([3.6721, 3.0930])
Epoch : 10, Loss: 29.105247, params: tensor([ 0.2325, -0.0162]), grad: tensor([1.4803, 3.0544])
Epoch : 11, Loss: 29.104168,